In [2]:
import torch
import cv2
from torchvision import transforms

In [3]:
from ref import EViT

In [4]:
ls ../../models/checkpoints/

model.pth        model_v_2_0.pth  model_v_5_0.pth  model_v_8_0.pth
model_v_0_0.pth  model_v_3_0.pth  model_v_6_0.pth  model_v_9_0.pth
model_v_1_0.pth  model_v_4_0.pth  model_v_7_0.pth


In [5]:
model = EViT(num_classes=81)

state_dict = torch.load(r"../../models/checkpoints/model.pth",map_location="cpu")

In [6]:
# Load trained model
model.load_state_dict(state_dict)

<All keys matched successfully>

In [7]:
model.eval()

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

EViT(
  (encoder): EViTEncoder(
    (patch_embeds): ModuleList(
      (0): OverlapPatchEmbedding(
        (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
        (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      )
      (1): OverlapPatchEmbedding(
        (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
      (2): OverlapPatchEmbedding(
        (proj): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (3): OverlapPatchEmbedding(
        (proj): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
    (blocks): ModuleList(
      (0): ModuleList(
        (0-2): 3 x TransformerBlock(
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine

In [8]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(model)
print(f"Total: {total/1e6:.2f}M | Trainable: {trainable/1e6:.2f}M")

Total: 17.40M | Trainable: 17.40M


In [9]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model = model.to(device)
model.eval()

# Color palette for 81 classes
colors = np.random.randint(0,255,(81,3))

# Webcam
cap = cv2.VideoCapture(0)

while True:
    
    ret, frame = cap.read()
    if not ret:
        break
    
    # Keep original frame size
    orig_h, orig_w = frame.shape[:2]
    
    # Resize for model
    img = cv2.resize(frame,(512,512))
    
    # Convert BGR -> RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Normalize
    img = img / 255.0
    
    # HWC -> CHW
    img = img.transpose(2,0,1)
    
    # Tensor
    img = torch.tensor(img,dtype=torch.float32).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        output = model(img)
    
    # Segmentation prediction
    pred = torch.argmax(output,dim=1)
    mask = pred.squeeze().cpu().numpy()
    
    mask_rgb = colors[mask].astype(np.uint8)
    mask_rgb = cv2.resize(mask_rgb,(orig_w,orig_h))
    
    # Overlay segmentation on frame
    overlay = cv2.addWeighted(frame,0.6,mask_rgb.astype(np.uint8),0.4,0)
    
    # Display in Jupyter
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    
    display(plt.gcf())
    clear_output(wait=True)
    plt.close()

cap.release()

KeyboardInterrupt: 

In [10]:
torch.save(model, "model_full.pth")